# 01 — Data Source, Quality, and EDA

**Objective:** Make the raw source explicit, validate it, clean implausible rows, and understand the target.

In [1]:
from pathlib import Path
import os, sys
import pandas as pd
import plotly.express as px

_cwd = Path.cwd().resolve()
ROOT = _cwd.parent if _cwd.name == "dev" else _cwd
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MPLCONFIGDIR", "/tmp/credit-risk-lab-matplotlib")

from credit_risk_lab.config.settings import settings
print(f"Project root: {settings.project_root}")

Project root: /Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab


## 1. Explicit CSV configuration

The path comes from `configs/settings.yaml`; the repository receives it explicitly with parsing options.

In [2]:
from credit_risk_lab.infrastructure.data_sources import CSVDataSourceConfig, CSVDatasetRepository

raw_source = CSVDataSourceConfig(
    path=settings.raw_data_path,
    sep=settings.raw_data_sep,
    encoding=settings.raw_data_encoding,
)
repository = CSVDatasetRepository(raw_source)
print({
    "configured_path": str(settings.raw_data_path_config),
    "resolved_path": str(repository.csv_path),
    "read_options": raw_source.read_kwargs(),
})
raw_df = repository.load()
raw_df.head()

{'configured_path': 'data/raw/loan_data.csv', 'resolved_path': '/Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab/data/raw/loan_data.csv', 'read_options': {'sep': ',', 'encoding': 'utf-8'}}
2026-07-11 09:51:50 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:51 - Chargement du fichier : /Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab/data/raw/loan_data.csv


2026-07-11 09:51:50 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:59 - Dataset chargé (45000 lignes, 14 colonnes)


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


## 2. Structural and business quality

The quality gate reports problems without silently modifying the source.

In [3]:
import pandas as pd
from credit_risk_lab.infrastructure.data_quality import build_quality_report, clean_implausible_rows

quality = build_quality_report(raw_df)
clean_df = clean_implausible_rows(raw_df)
pd.DataFrame([{**quality.as_dict(), "clean_rows": len(clean_df), "removed_rows": len(raw_df)-len(clean_df)}])

,rows,columns,duplicate_rows,missing_values,invalid_age_rows,invalid_experience_rows,clean_rows,removed_rows
0,45000,14,0,0,7,7,44993,7


## 3. Target and numerical ranges

The positive class is the synthetic risk class. Its exact business definition must be replaced before real use.

In [4]:
import plotly.express as px

target = raw_df[settings.target_column].value_counts().rename_axis("class").reset_index(name="rows")
target["rate"] = target["rows"] / len(raw_df)
display(target)
px.bar(target, x="class", y="rows", text="rate", title="Target distribution", template="plotly_white").show()
raw_df.select_dtypes("number").agg(["min", "median", "max"]).T

,class,rows,rate
0,0,35000,0.777778
1,1,10000,0.222222


,min,median,max
person_age,20.00,26.00,144.00
person_income,8000.00,67048.00,7200766.00
person_emp_exp,0.00,4.00,125.00
loan_amnt,500.00,8000.00,35000.00
loan_int_rate,5.42,11.01,20.00
loan_percent_income,0.00,0.12,0.66
cb_person_cred_hist_length,2.00,4.00,30.00
credit_score,390.00,640.00,850.00
loan_status,0.00,0.00,1.00


## Conclusion

The source path, parsing options, quality problems, cleaning impact, and target balance are now explicit.